In [12]:
import os

from autogen import ConversableAgent, register_function

if "GROQ_API_KEY" not in os.environ:
    raise EnvironmentError("Please set the GROQ_API_KEY environment variable before running this notebook.")

llm_config = {
    "config_list": [
        {
            "model": "llama-3.1-8b-instant",
            "base_url": "https://api.groq.com/openai/v1",
            "api_key": os.environ["GROQ_API_KEY"],
        }
    ]
}

print("Groq LLM config is ready.")

Groq LLM config is ready.


In [4]:

import yfinance as yf

def get_stock_price(symbol: str):
    try:
        normalized_symbol = symbol.strip().upper()
        ticker = yf.Ticker(normalized_symbol)
        history = ticker.history(period="1d", interval="1m")

        if history.empty or "Close" not in history:
            return {
                "symbol": normalized_symbol,
                "message": "Symbol not found or no market data available.",
                "source": "Yahoo Finance"
            }

        price = float(history["Close"].dropna().iloc[-1])

        info = getattr(ticker, "fast_info", {}) or {}
        currency = info.get("currency") or "USD"

        return {
            "symbol": normalized_symbol,
            "price": price,
            "currency": currency,
            "source": "Yahoo Finance"
        }
    except Exception as e:
        return {
            "symbol": symbol.strip().upper(),
            "message": f"Network or data error: {str(e)}",
            "source": "Yahoo Finance"
        }

In [13]:
assistant = ConversableAgent(
    name="assistant",
    system_message=(
        "You are an AI assistant. Decide when to call the stock tool to fetch stock prices. "
        "Use the tool when needed and explain the results clearly."
    ),
    llm_config=llm_config,
)

user_proxy = ConversableAgent(
    name="user_proxy",
    llm_config=False,
    human_input_mode="NEVER",
)

print("Agents created.")

Agents created.


In [14]:
register_function(
    get_stock_price,
    caller=assistant,
    executor=user_proxy,
    name="get_stock_price",
    description="Get a stock/asset price by symbol (TSLA, AAPL, BTC).",
)

print("Tool registered successfully.")

Tool registered successfully.


In [22]:
import json
import logging
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from pathlib import Path

# Optional: reduce verbose AutoGen logs in notebook output.
logging.getLogger("autogen").setLevel(logging.ERROR)

# Capture verbose stdout/stderr so notebook output stays clean.
captured_stdout = StringIO()
captured_stderr = StringIO()
with redirect_stdout(captured_stdout), redirect_stderr(captured_stderr):
    chat_result = user_proxy.initiate_chat(
        assistant,
        message="What is the price of Tesla stock?",
        clear_history=True,
        max_turns=2,
        silent=True,
    )

history = getattr(chat_result, "chat_history", [])

# Save full conversation + captured verbose logs to a text file.
log_path = Path("autogen_full_chat_log.txt")
with log_path.open("w", encoding="utf-8") as f:
    f.write("=== Captured AutoGen Stdout ===\n")
    f.write(captured_stdout.getvalue())
    f.write("\n\n=== Captured AutoGen Stderr ===\n")
    f.write(captured_stderr.getvalue())
    f.write("\n\n=== Chat History ===\n")
    for i, msg in enumerate(history, start=1):
        role = msg.get("role", "unknown")
        name = msg.get("name")
        content = msg.get("content")
        f.write(f"[{i}] role={role}")
        if name:
            f.write(f", name={name}")
        f.write("\n")
        if isinstance(content, (dict, list)):
            f.write(json.dumps(content, indent=2, ensure_ascii=False))
        else:
            f.write(str(content))
        f.write("\n\n")

# Optional: show latest tool result.
tool_result = None
for msg in reversed(history):
    if msg.get("role") == "tool" and msg.get("content") is not None:
        tool_result = msg["content"]
        break

# Show only the final meaningful assistant answer.
final_answer = None
for msg in reversed(history):
    content = msg.get("content")
    if not content or not isinstance(content, str):
        continue
    if msg.get("name") == "assistant" or (msg.get("role") == "assistant" and content.lower() != "none"):
        final_answer = content
        break

if tool_result is not None:
    print("Tool result:", tool_result)
print("Final assistant answer:", final_answer if final_answer else "No assistant answer found.")
print(f"Full chat log saved to: {log_path.resolve()}")


>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION get_stock_price...
Call ID: vahj7p64e
Input arguments: {'symbol': 'TSLA'}

>>>>>>>> EXECUTED FUNCTION get_stock_price...
Call ID: vahj7p64e
Input arguments: {'symbol': 'TSLA'}
Output:
{'symbol': 'TSLA', 'price': 400.6600036621094, 'currency': 'USD', 'source': 'Yahoo Finance'}

>>>>>>>> USING AUTO REPLY...

>>>>>>>> TERMINATING RUN (db07be99-04ab-4b2c-ac9a-a484fdd1d121): Maximum turns (2) reached
Tool result: {'symbol': 'TSLA', 'price': 400.6600036621094, 'currency': 'USD', 'source': 'Yahoo Finance'}
Final assistant answer: The current price of Tesla stock is $400.66 USD, according to Yahoo Finance.
Full chat log saved to: F:\New folder\ai\autogen\tool_call\autogen_full_chat_log.txt
